# LeetCode 2535 — Difference Between Element Sum and Digit Sum of an Array bằng Python

Link đề: https://leetcode.com/problems/difference-between-element-sum-and-digit-sum-of-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def differenceOfSum(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **element sum** là tổng các số trong mảng; **digit sum** là tổng **tất cả chữ số** của **tất cả** số trong mảng; trả về `|element sum - digit sum|`.
2. Tách chữ số của một số nguyên bằng `% 10` và `// 10` (hoặc `divmod`), và bằng cách chuyển sang chuỗi `str(x)`.
3. Nhận ra tính chất toán học: `x - (tổng chữ số của x)` luôn `>= 0` và **chia hết cho 9**, nên `abs` ở đề thực ra không cần thiết.
4. Viết 4 cách: tách chữ số bằng `%` và `//`, bằng chuỗi, bằng bảng tính trước `ds[x] = ds[x // 10] + x % 10`, và bằng công thức vị trí `d * (10^i - 1)`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: lấy tổng chữ số của **tổng** mảng, sửa biến lặp rồi dùng lại, đếm **số** chữ số thay vì **tổng** chữ số.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums`.

- **Element sum**: tổng tất cả các phần tử của `nums`.
- **Digit sum**: tổng tất cả **chữ số** xuất hiện trong **mọi** phần tử của `nums` (ví dụ số `15` đóng góp `1 + 5 = 6`).

Trả về **giá trị tuyệt đối** của hiệu `element sum - digit sum`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,15,6,3]` | 9 | element sum `1+15+6+3 = 25`; digit sum `1+1+5+6+3 = 16`; `|25 - 16| = 9` |
| 2 | `[1,2,3,4]` | 0 | element sum = `10`, digit sum = `10` (mọi số có một chữ số) |

Constraints: `1 <= nums.length <= 2000`; `1 <= nums[i] <= 2000`.

In [1]:
nums = [1, 15, 6, 3]
element_sum = sum(nums)
digits = [int(ch) for x in nums for ch in str(x)]
print("các chữ số:", digits)
print("element sum =", element_sum, "| digit sum =", sum(digits))
print("kết quả =", abs(element_sum - sum(digits)))

các chữ số: [1, 1, 5, 6, 3]
element sum = 25 | digit sum = 16
kết quả = 9


## 3. Kiến thức nền

### 3.1. `%` và `//` — lấy chữ số cuối và bỏ chữ số cuối

`x % 10` cho **chữ số cuối** của `x` (`15 % 10 = 5`). `x // 10` là chia nguyên, **bỏ** chữ số cuối (`15 // 10 = 1`). Lặp lại cho tới khi `x` thành `0` ta lấy được mọi chữ số từ phải sang trái.

In [2]:
x = 1234
while x > 0:
    print("chữ số cuối:", x % 10, "| phần còn lại:", x // 10)
    x //= 10

chữ số cuối: 4 | phần còn lại: 123
chữ số cuối: 3 | phần còn lại: 12
chữ số cuối: 2 | phần còn lại: 1
chữ số cuối: 1 | phần còn lại: 0


### 3.2. `divmod(a, b)` — thương và dư cùng lúc

`divmod(x, 10)` trả về cặp `(x // 10, x % 10)`. Dùng được trong `while x: x, d = divmod(x, 10)`, gọn hơn hai phép riêng lẻ.

In [3]:
print(divmod(15, 10), divmod(7, 10), divmod(2000, 10))
x, total = 305, 0
while x:
    x, d = divmod(x, 10)
    total += d
print(total)

(1, 5) (0, 7) (200, 0)
8


### 3.3. Duyệt từng ký tự của `str(x)`

`str(15)` là `"15"`; duyệt `for ch in "15"` cho `'1'`, `'5'` (là **chuỗi**, không phải số). `int(ch)` đổi lại thành số. Cách này ngắn nhưng tạo chuỗi trung gian.

In [4]:
print([int(ch) for ch in str(15)])
print(sum(int(ch) for ch in str(2000)))
print(type(str(15)[0]), type(int(str(15)[0])))

[1, 5]
2
<class 'str'> <class 'int'>


### 3.4. Biến lặp trong `for` và việc sửa nó

`for x in nums:` gán `x` từng phần tử. Nếu ta **sửa** `x` (ví dụ `x //= 10` để tách chữ số) thì `x` mất giá trị gốc; sau vòng `while` nó thành `0`. Muốn dùng lại giá trị gốc phải **cộng nó trước** hoặc dùng biến tạm (`y = x`). Đây là nguồn lỗi hay gặp (xem mục 13).

In [5]:
nums = [15, 6]
for x in nums:
    y = x                      # biến tạm để giữ nguyên x
    while y > 0:
        y //= 10
    print("x vẫn là", x, "| y sau vòng lặp là", y)
print(nums)                    # list gốc không đổi

x vẫn là 15 | y sau vòng lặp là 0
x vẫn là 6 | y sau vòng lặp là 0
[15, 6]


### 3.5. Quy hoạch động nhỏ: bảng `ds[x] = ds[x // 10] + x % 10`

Tổng chữ số của `x` = tổng chữ số của `x // 10` cộng chữ số cuối `x % 10`. Vì `x // 10 < x` ta có thể điền bảng `ds` từ `0` lên đến giá trị lớn nhất trong mảng. Mỗi `ds[x]` tốn `O(1)`, rồi mọi số trong mảng tra bảng. Hiệu quả khi có nhiều số lặp lại hoặc nhiều truy vấn.

In [6]:
M_ = 30
ds = [0] * (M_ + 1)
for v in range(1, M_ + 1):
    ds[v] = ds[v // 10] + v % 10
print(ds)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 3]


### 3.6. Số và tổng chữ số cùng số dư khi chia 9

Vì `10 ≡ 1 (mod 9)`, mọi luỹ thừa `10^i ≡ 1 (mod 9)`; nên `x ≡ (tổng chữ số của x) (mod 9)`. Suy ra `x - ds(x)` luôn **chia hết cho 9**; và vì mỗi chữ số `d` ở vị trí `i` đóng góp `d * 10^i >= d`, ta có `x >= ds(x)`. Do đó `x - ds(x) >= 0` và `abs` trong đề **không bao giờ đổi dấu** (nhưng vẫn nên viết cho đúng đề).

In [7]:
for x in [1, 15, 99, 2000, 1999]:
    s = sum(int(c) for c in str(x))
    print(f"x={x:>4}: digit sum={s:>2}, x - ds = {x - s:>4}, chia hết 9? {(x - s) % 9 == 0}")

x=   1: digit sum= 1, x - ds =    0, chia hết 9? True
x=  15: digit sum= 6, x - ds =    9, chia hết 9? True
x=  99: digit sum=18, x - ds =   81, chia hết 9? True
x=2000: digit sum= 2, x - ds = 1998, chia hết 9? True
x=1999: digit sum=28, x - ds = 1971, chia hết 9? True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm đúng như đề: tính hai tổng rồi lấy chênh lệch.

Ví dụ 1 `[1, 15, 6, 3]`:

```
element sum = 1 + 15 + 6 + 3 = 25
chữ số     : 1 | 1 5 | 6 | 3  -> 1 + 1 + 5 + 6 + 3 = 16
|25 - 16| = 9
```

Quan sát: số **một chữ số** có "giá trị" bằng "tổng chữ số" nên không tạo chênh lệch (`1`, `6`, `3` đều bằng chính nó). Chênh lệch chỉ đến từ các số **từ hai chữ số trở lên**: `15` cho `15 - 6 = 9`. Do đó đáp án = tổng các `x - ds(x)` trên các số `x` của mảng, mà không cần giữ hai tổng riêng.

## 5. Cách 1 — Brute Force / baseline

Với mỗi số, tách chữ số bằng `% 10` và `// 10` (làm trên **bản sao** `y`), cộng vào `digit_sum`; đồng thời cộng số vào `element_sum`. Cuối cùng trả `abs(element_sum - digit_sum)`.

Nút thắt: không có — mỗi số có tối đa 4 chữ số (`<= 2000`) nên tổng chi phí `O(n * D)` với `D` là số chữ số. Đây đã là tối ưu về bậc; các cách sau chỉ khác về phong cách.

In [8]:
class SolutionDigits:
    def differenceOfSum(self, nums):
        element_sum = 0
        digit_sum = 0
        for x in nums:
            element_sum += x
            y = x                      # làm việc trên bản sao
            while y > 0:
                digit_sum += y % 10
                y //= 10
        return abs(element_sum - digit_sum)

print(SolutionDigits().differenceOfSum([1, 15, 6, 3]))
print(SolutionDigits().differenceOfSum([1, 2, 3, 4]))

9
0


**Độ phức tạp:** Time `O(n * D)` với `D <= 4` (số chữ số), Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Gộp hai tổng thành **một vòng lặp**: `answer = Σ (x - ds(x))`, vì

`Σ x - Σ ds(x) = Σ (x - ds(x))`.

Mỗi hạng tử `x - ds(x)` luôn `>= 0` (và chia hết cho 9), nên cả tổng `>= 0` và `abs(...)` là dư thừa — ta có thể bỏ nó mà kết quả vẫn đúng.

**Quy tắc "nếu… thì…":**

- Nếu `x < 10` → `x - ds(x) = 0`, bỏ qua được (đỡ phải tách chữ số).
- Nếu `x >= 10` → tách chữ số và cộng `x - ds(x)`.
- Có thể tính `x - ds(x)` trực tiếp bằng công thức vị trí: chữ số `d` ở vị trí `i` (từ phải, `i = 0` là hàng đơn vị) đóng góp `d * (10^i - 1)`.

Invariant: sau khi xử lý `k` phần tử, biến tích luỹ bằng `Σ_{j<k} (nums[j] - ds(nums[j]))`.

In [9]:
def ds_of(x):
    return sum(int(c) for c in str(x))

nums = [1, 15, 6, 3]
print([x - ds_of(x) for x in nums], "->", sum(x - ds_of(x) for x in nums))

[0, 9, 0, 0] -> 9


## 7. Các cách tiếp theo

Ta có 4 cách: (1) tách chữ số bằng `%` và `//`; (2) chuyển sang chuỗi; (3) bảng tính trước `ds[x]`; (4) công thức vị trí `d * (10^i - 1)`.

### Cách 2 — Chuyển sang chuỗi

`sum(map(int, str(x)))` cho tổng chữ số. Toàn bộ bài giải trong vài dòng. Đánh đổi: tạo chuỗi trung gian cho mỗi số (nhỏ, không đáng kể ở đây).

In [10]:
class SolutionString:
    def differenceOfSum(self, nums):
        element_sum = sum(nums)
        digit_sum = sum(int(ch) for x in nums for ch in str(x))
        return abs(element_sum - digit_sum)

print(SolutionString().differenceOfSum([1, 15, 6, 3]))
print(SolutionString().differenceOfSum([1, 2, 3, 4]))

9
0


Độ phức tạp Cách 2: Time `O(n * D)`, Space `O(D)` (chuỗi tạm của một số).

### Cách 3 — Bảng tính trước `ds[x]`

Vì `nums[i] <= 2000`, dựng `ds[0..max(nums)]` một lần bằng `ds[v] = ds[v // 10] + v % 10`; sau đó tổng chữ số của mỗi số chỉ là một lần tra bảng. Phù hợp khi có nhiều truy vấn hoặc mảng rất dài.

In [11]:
class SolutionTable:
    def differenceOfSum(self, nums):
        top = max(nums)
        ds = [0] * (top + 1)
        for v in range(1, top + 1):
            ds[v] = ds[v // 10] + v % 10
        return abs(sum(nums) - sum(ds[x] for x in nums))

print(SolutionTable().differenceOfSum([1, 15, 6, 3]))
print(SolutionTable().differenceOfSum([1, 2, 3, 4]))

9
0


Độ phức tạp Cách 3: Time `O(max(nums) + n)`, Space `O(max(nums))`. Chỉ đáng dùng khi `max(nums)` nhỏ (ở đây `<= 2000`).

### Cách 4 — Công thức vị trí `d * (10^i - 1)`

Số `x = Σ d_i * 10^i`, còn `ds(x) = Σ d_i`. Do đó `x - ds(x) = Σ d_i * (10^i - 1)`. Hàng đơn vị (`i = 0`) đóng góp `0`; chỉ các hàng chục, trăm, nghìn mới tạo chênh lệch. Cộng dồn trực tiếp không cần giữ hai tổng.

In [12]:
class SolutionPositional:
    def differenceOfSum(self, nums):
        total = 0
        for x in nums:
            place = 1                  # 10^i
            while x > 0:
                x, d = divmod(x, 10)
                total += d * (place - 1)
                place *= 10
        return total                   # luôn >= 0, không cần abs

print(SolutionPositional().differenceOfSum([1, 15, 6, 3]))
print(SolutionPositional().differenceOfSum([1, 2, 3, 4]))

9
0


Độ phức tạp Cách 4: Time `O(n * D)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 1 với `nums = [1, 15, 6, 3]`:

| `x` | các chữ số tách được | `element_sum` | `digit_sum` |
|---|---|---|---|
| 1 | 1 | 1 | 1 |
| 15 | 5, 1 | 16 | 7 |
| 6 | 6 | 22 | 13 |
| 3 | 3 | 25 | 16 |

Cuối cùng `|25 - 16| = 9`.

In [13]:
def trace_digits(nums):
    element_sum = digit_sum = 0
    for x in nums:
        element_sum += x
        y = x
        got = []
        while y > 0:
            got.append(y % 10)
            digit_sum += y % 10
            y //= 10
        print(f"x={x:>4}: chữ số (phải -> trái) {got} | element_sum={element_sum}, digit_sum={digit_sum}")
    print(f"|{element_sum} - {digit_sum}| = {abs(element_sum - digit_sum)}")
    return abs(element_sum - digit_sum)

trace_digits([1, 15, 6, 3])
trace_digits([2000, 99, 7])

x=   1: chữ số (phải -> trái) [1] | element_sum=1, digit_sum=1
x=  15: chữ số (phải -> trái) [5, 1] | element_sum=16, digit_sum=7
x=   6: chữ số (phải -> trái) [6] | element_sum=22, digit_sum=13
x=   3: chữ số (phải -> trái) [3] | element_sum=25, digit_sum=16
|25 - 16| = 9
x=2000: chữ số (phải -> trái) [0, 0, 0, 2] | element_sum=2000, digit_sum=2
x=  99: chữ số (phải -> trái) [9, 9] | element_sum=2099, digit_sum=20
x=   7: chữ số (phải -> trái) [7] | element_sum=2106, digit_sum=27
|2106 - 27| = 2079


2079

## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def differenceOfSum(self, nums: List[int]) -> int:
        element_sum = sum(nums)
        digit_sum = 0
        for x in nums:
            while x > 0:
                x, d = divmod(x, 10)
                digit_sum += d
        return abs(element_sum - digit_sum)

print(Solution().differenceOfSum([1, 15, 6, 3]), Solution().differenceOfSum([1, 2, 3, 4]))

9 0


## 10. Vì sao thuật toán đúng

- **Tách chữ số đúng:** với `x > 0`, `x % 10` là chữ số hàng đơn vị và `x // 10` là số còn lại sau khi bỏ chữ số đó (`x = 10 * (x // 10) + x % 10`). Lặp tới khi `x = 0` thì đã lấy hết các chữ số, mỗi chữ số một lần. `x >= 1` nên không có trường hợp `x = 0` (chữ số `0` đơn lẻ).
- **Hai tổng độc lập:** `element_sum = Σ x` và `digit_sum = Σ ds(x)`; đề yêu cầu đúng `|element_sum - digit_sum|`.
- **`abs` dư thừa nhưng vô hại:** `x = Σ d_i 10^i >= Σ d_i = ds(x)` vì `10^i >= 1`; do đó `element_sum >= digit_sum` và hiệu `>= 0`.
- **Cách 4:** `x - ds(x) = Σ d_i (10^i - 1)`, hạng tử `i = 0` bằng `0`; cộng lại chính là `Σ (x - ds(x))`.
- **Cách 3:** `ds[v] = ds[v // 10] + v % 10` đúng vì chữ số của `v` là chữ số của `v // 10` cộng chữ số cuối; `ds[0] = 0` là cơ sở.
- **Kiểm tra chéo bằng mod 9:** kết quả luôn chia hết cho 9 — một "dấu hiệu" để tự kiểm tra đáp án.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `D` = số chữ số tối đa của một phần tử (`D <= 4` với `nums[i] <= 2000`), `V = max(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. `%` và `//` | `O(n * D)` | `O(1)` |
| 2. Chuyển sang chuỗi | `O(n * D)` | `O(D)` |
| 3. Bảng `ds[x]` | `O(V + n)` | `O(V)` |
| 4. Công thức vị trí | `O(n * D)` | `O(1)` |

Vì `D` là hằng số nhỏ (`<= 4`), cả bốn cách thực tế là `O(n)`; mọi cách phải đọc từng phần tử nên không thể nhanh hơn.

## 12. Bộ test

Test 1: ví dụ đề và các edge case (một phần tử, toàn số một chữ số, `2000`, `1999`, `10`, `99`).

In [15]:
impls = [SolutionDigits(), SolutionString(), SolutionTable(), SolutionPositional(), Solution()]
tests = [
    ([1, 15, 6, 3], 9),
    ([1, 2, 3, 4], 0),
    ([1], 0),                        # nhỏ nhất
    ([9], 0),                        # một chữ số
    ([10], 9),                       # 10 - (1+0) = 9
    ([99], 81),                      # 99 - 18
    ([100], 99),                     # 100 - 1
    ([2000], 1998),                  # 2000 - 2: biên trên
    ([1999], 1999 - 28),             # 1999 - (1+9+9+9)
    ([10, 10, 10], 27),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9], 0),
    ([11, 22, 33], 54),              # 11-2 = 9, 22-4 = 18, 33-6 = 27
    ([2000, 2000, 2000], 5994),
    ([101, 202, 303], 99 + 198 + 297),
]
for nums, exp in tests:
    assert 1 <= len(nums) <= 2000 and all(1 <= v <= 2000 for v in nums)
    for im in impls:
        assert im.differenceOfSum(list(nums)) == exp, (type(im).__name__, nums, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 14


Test 2: ngẫu nhiên có seed, đúng constraints (`1 <= n <= 2000`, `1 <= nums[i] <= 2000`); oracle độc lập dùng chuỗi và tổng bằng `abs` theo đúng đề.

In [16]:
import random
random.seed(2535)

def oracle(nums):
    element_sum = sum(nums)
    digit_sum = sum(int(ch) for x in nums for ch in str(x))
    return abs(element_sum - digit_sum)

for t in range(800):
    n = random.randint(1, 2000) if t % 50 == 0 else random.randint(1, 30)
    hi_val = random.choice([9, 99, 999, 2000])
    nums = [random.randint(1, hi_val) for _ in range(n)]
    assert 1 <= len(nums) <= 2000 and all(1 <= v <= 2000 for v in nums)
    exp = oracle(nums)
    for im in impls:
        assert im.differenceOfSum(list(nums)) == exp, (type(im).__name__, nums[:10])
print("800 test ngẫu nhiên OK")

800 test ngẫu nhiên OK


Test 3: tính chất — kết quả `>= 0` và **chia hết cho 9**; hoán vị mảng không đổi kết quả; không sửa input; mảng toàn số một chữ số cho `0`.

In [17]:
random.seed(35)
for _ in range(500):
    n = random.randint(1, 40)
    nums = [random.randint(1, 2000) for _ in range(n)]
    base = Solution().differenceOfSum(nums)
    assert base >= 0 and base % 9 == 0
    shuffled = list(nums)
    random.shuffle(shuffled)
    assert Solution().differenceOfSum(shuffled) == base
    snapshot = list(nums)
    Solution().differenceOfSum(nums)
    assert nums == snapshot
    small = [random.randint(1, 9) for _ in range(n)]
    assert Solution().differenceOfSum(small) == 0
print("tính chất (không âm, chia hết 9, bất biến theo hoán vị, không sửa input, số một chữ số -> 0): OK")

tính chất (không âm, chia hết 9, bất biến theo hoán vị, không sửa input, số một chữ số -> 0): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: lấy tổng chữ số của **tổng** mảng thay vì của từng phần tử

In [18]:
def wrong_digits_of_total(nums):
    element_sum = sum(nums)
    digit_sum = sum(int(ch) for ch in str(element_sum))      # SAI: tổng chữ số của 25, không phải của từng phần tử
    return abs(element_sum - digit_sum)

nums = [1, 15, 6, 3]
print("sai :", wrong_digits_of_total(nums))
print("đúng:", Solution().differenceOfSum(nums))
assert wrong_digits_of_total(nums) != Solution().differenceOfSum(nums)

sai : 18
đúng: 9


Vì sao sai: đề cần tổng chữ số của **mọi phần tử** (`1, 1, 5, 6, 3`), không phải chữ số của số `25`. **Sửa:** tách chữ số **từng phần tử** rồi cộng.

### Lỗi 2: sửa biến lặp rồi dùng lại nó để cộng element sum

In [19]:
def wrong_mutate(nums):
    element_sum = 0
    digit_sum = 0
    for x in nums:
        while x > 0:
            digit_sum += x % 10
            x //= 10                  # x bị phá hủy, cuối vòng while x == 0
        element_sum += x              # SAI: cộng x (đã thành 0) thay vì giá trị gốc
    return abs(element_sum - digit_sum)

nums = [1, 15, 6, 3]
print("sai :", wrong_mutate(nums))
print("đúng:", Solution().differenceOfSum(nums))
assert wrong_mutate(nums) != Solution().differenceOfSum(nums)

sai : 16
đúng: 9


Vì sao sai: sau vòng `while`, `x` đã bị chia dần về `0` nên `element_sum` luôn là `0` và ta trả `digit_sum`. **Sửa:** cộng `x` **trước** vòng `while`, hoặc dùng biến phụ `y = x`, hoặc tính `sum(nums)` riêng.

### Lỗi 3: đếm **số** chữ số thay vì **tổng** chữ số

In [20]:
def wrong_count_digits(nums):
    element_sum = sum(nums)
    digit_sum = sum(len(str(x)) for x in nums)               # SAI: đếm số chữ số (1,2,1,1) thay vì cộng chữ số
    return abs(element_sum - digit_sum)

nums = [1, 15, 6, 3]
print("sai :", wrong_count_digits(nums))
print("đúng:", Solution().differenceOfSum(nums))
assert wrong_count_digits(nums) != Solution().differenceOfSum(nums)

sai : 20
đúng: 9


Vì sao sai: `len(str(15)) = 2` (số chữ số) khác `1 + 5 = 6` (tổng chữ số). **Sửa:** `sum(int(ch) for ch in str(x))`.

### Lỗi 4 (bonus): quên đổi ký tự thành số

`sum(str(x))` hoặc cộng `ch` mà không `int(ch)` gây `TypeError` (cộng chuỗi với số). Luôn `int(ch)` khi duyệt `str(x)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng chữ số" → vòng lặp `while x: x, d = divmod(x, 10)` hoặc `sum(map(int, str(x)))`.
- Hai tổng cần trừ nhau → có thể gộp thành **một** tổng `Σ (x - ds(x))`, bớt biến.
- Tính chất mod 9: `x ≡ ds(x) (mod 9)` → kết quả luôn chia hết cho 9; dùng để tự kiểm tra.
- Số một chữ số không tạo chênh lệch → có thể bỏ qua cho nhanh.
- **Cẩn thận biến lặp bị sửa**: dùng biến tạm khi tách chữ số.
- `1 <= nums[i] <= 2000` nhỏ → các cách "đơn giản" (chuỗi) đã đủ; đừng tối ưu quá mức.

## 15. Biến thể của bài

1. Tính **chênh lệch giữa hai tổng chữ số** của hai mảng.
2. Trả về số phần tử mà `x` chia hết cho tổng chữ số của nó (Harshad / Niven numbers).
3. Cho `nums[i]` tới `10^18`: vẫn tách chữ số được, nhưng nhớ tích luỹ bằng số nguyên Python (không tràn).
4. Tổng chữ số lặp lại tới khi còn một chữ số (**digital root**, LeetCode 258).
5. Đếm số `x` trong `1..N` mà `x - ds(x) >= k` (phụ thuộc cấu trúc chữ số, cần DP chữ số).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `%` và `//` | tách chữ số từ phải sang trái | `O(n * D)` | `O(1)` | **chuẩn**, không cần chuỗi |
| 2. Chuyển sang chuỗi | `int(ch)` cho mỗi ký tự | `O(n * D)` | `O(D)` | ngắn nhất, dễ đọc |
| 3. Bảng `ds[x]` | `ds[x // 10] + x % 10` | `O(V + n)` | `O(V)` | nhiều truy vấn, `V` nhỏ |
| 4. Công thức vị trí | `d * (10^i - 1)` | `O(n * D)` | `O(1)` | hiểu bản chất, bỏ `abs` |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `differenceOfSum([20, 5, 100])` bằng bao nhiêu?

**Bài 2 (tính tay):** với `[37, 8, 120]`: tính element sum, digit sum rồi hiệu.

**Bài 3 (điền TODO):** hoàn thành `digit_sum(x)` tổng chữ số của một số nguyên dương.

**Bài 4 (tự viết lại):** viết `difference_no_abs(nums)` không dùng `abs`, dựa vào `Σ (x - ds(x))`.

**Bài 5 (mini challenge):** viết `count_harshad(nums)` đếm các số `x` chia hết cho `digit_sum(x)` (số Harshad).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().differenceOfSum([20, 5, 100]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def digit_sum(x):
    total = 0
    # TODO: lặp trong khi x > 0: total += x % 10; x //= 10
    return total

117


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** hai tổng: tổng các số và tổng **tất cả chữ số** của các số đó.

**Mức 2 (vừa):** Bài 3: `while x > 0: total += x % 10; x //= 10`. Bài 4: `sum(x - digit_sum(x) for x in nums)`. Bài 5: `x % digit_sum(x) == 0`.

**Mức 3 (gần đáp án):** Bài 1: element sum `125`; digit sum `2 + 5 + 1 = 8` → `117`. Bài 2: element sum `165`; digit sum `3+7+8+1+2+0 = 21` → `144`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: 20 + 5 + 100 = 125 ; chữ số 2+0+5+1+0+0 = 8 -> 117
assert Solution().differenceOfSum([20, 5, 100]) == 117

# Bài 2: [37, 8, 120] -> 165 ; digit sum 3+7+8+1+2+0 = 21 -> 144
assert sum([37, 8, 120]) == 165
assert Solution().differenceOfSum([37, 8, 120]) == 144

# Bài 3
def digit_sum(x):
    total = 0
    while x > 0:
        total += x % 10
        x //= 10
    return total
assert digit_sum(15) == 6 and digit_sum(2000) == 2 and digit_sum(1999) == 28

# Bài 4
def difference_no_abs(nums):
    return sum(x - digit_sum(x) for x in nums)
assert difference_no_abs([1, 15, 6, 3]) == 9
assert difference_no_abs([1, 2, 3, 4]) == 0
assert difference_no_abs([2000]) == 1998

# Bài 5
def count_harshad(nums):
    return sum(1 for x in nums if x % digit_sum(x) == 0)
assert count_harshad([18, 19, 20, 21]) == 3          # 18 (9), 20 (2), 21 (3) chia hết; 19 không
assert count_harshad([11, 13]) == 0
assert count_harshad([1, 2, 3]) == 3
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tổng chữ số của từng phần tử":**

```python
element_sum = sum(nums)
digit_sum = 0
for x in nums:
    while x > 0:
        x, d = divmod(x, 10)     # d là chữ số cuối
        digit_sum += d
return abs(element_sum - digit_sum)
```

**Checklist trước khi submit:**

- [ ] `element_sum` tính trên mảng gốc (không bị phá vì sửa biến lặp)?
- [ ] Tổng chữ số tính cho **từng phần tử**, không phải cho tổng cả mảng?
- [ ] Cộng **giá trị chữ số**, không phải số lượng chữ số (`len(str(x))`)?
- [ ] Nếu duyệt `str(x)` thì đã `int(ch)`?
- [ ] Đã thử `[1]`, `[10]`, `[2000]`, mảng toàn số một chữ số (kết quả `0`)?
- [ ] Kết quả chia hết cho 9 (tự kiểm tra) và đối chiếu các cách bằng test ngẫu nhiên có seed?
- [ ] Chữ ký đúng `differenceOfSum(self, nums: List[int]) -> int`?